In [1]:
import openeo

bbox = {
    "west": 18.9,
    "south": 48.8,
    "east": 19.3,
    "north": 49.1
}


In [2]:
#conn = openeo.connect("https://openeo.dataspace.copernicus.eu")
#conn.authenticate_oidc()

conn = openeo.connect("openeo.cloud").authenticate_oidc()


[#####################################] ⌛ Authorization pending

KeyboardInterrupt: 

In [ ]:
cube = conn.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=bbox,
    temporal_extent=["2015-01-01", "2025-12-31"],
    bands=["B04", "B08", "SCL"],
    max_cloud_cover=30
)

In [ ]:
cube = cube.filter_temporal(lambda t: (t.month >= 6) & (t.month <= 10))

In [ ]:
def mask_clouds(data):
    scl = data.band("SCL")
    
    # maskujeme oblaky, tiene, cirrus...
    mask = (
        (scl != 3) &  # cloud shadow
        (scl != 8) &  # medium cloud
        (scl != 9) &  # high cloud
        (scl != 10)   # cirrus
    )
    return data.mask(mask)

cube = cube.apply(mask_clouds)

In [ ]:
ndvi = cube.ndvi(nir="B08", red="B04")

In [ ]:
job = ndvi.create_job(
    out_format="GTiff",
    title="NDVI_monthly_Jun_Oct"
)

job.start_and_wait()

results = job.get_results()
results.download_files("ndvi_geotiff_output")

# ---------------------------------------------------------

In [9]:
# import necessary packages
import openeo
import openeo.processes as op
year_start = 2015
year_end = 2025

In [14]:


# connect with the backend
eoconn = openeo.connect(
        "openeo.dataspace.copernicus.eu"
        ).authenticate_oidc()
# Setup process parameters
bbox = {
        "west":  18.51635,
        "south": 48.78376,
        "east":  18.80255,
        "north": 49.04104
        }
date = f"{year_start}-05-05", f"{year_start}-12-31"

# Create a processing graph from the NDVI process using an active openEO connection
sentinel2_data_cube = eoconn.load_collection(
                      "SENTINEL2_L2A",
                       bands=["B04", "B08"]
                       )
sentinel2_data_cube = sentinel2_data_cube.filter_bbox(**bbox)
sentinel2_data_cube = sentinel2_data_cube.filter_temporal(date)
ndvi = sentinel2_data_cube.ndvi(red="B04",
    nir="B08")

Authenticated using refresh token.


In [15]:
ndvi_relevant_months_sections = []

month_start = "06"
month_end = "11"

for y in range(year_start, year_end+1):
        tmp_cube = ndvi.filter_temporal([f"{y}-{month_start}-01", f"{y}-{month_end}-01"])
        ndvi_relevant_months_sections.append(tmp_cube)

In [12]:
print(ndvi_relevant_months_sections[1])

{
  "process_graph": {
    "loadcollection1": {
      "process_id": "load_collection",
      "arguments": {
        "bands": [
          "B04",
          "B08"
        ],
        "id": "SENTINEL2_L2A",
        "spatial_extent": null,
        "temporal_extent": null
      }
    },
    "filterbbox1": {
      "process_id": "filter_bbox",
      "arguments": {
        "data": {
          "from_node": "loadcollection1"
        },
        "extent": {
          "west": 18.80255,
          "east": 18.51635,
          "north": 49.04104,
          "south": 48.78376
        }
      }
    },
    "filtertemporal1": {
      "process_id": "filter_temporal",
      "arguments": {
        "data": {
          "from_node": "filterbbox1"
        },
        "extent": [
          "2021-05-05",
          "2025-12-31"
        ]
      }
    },
    "ndvi1": {
      "process_id": "ndvi",
      "arguments": {
        "data": {
          "from_node": "filtertemporal1"
        }
      }
    },
    "filtertemporal2": 

In [16]:
# Test download one year
ndvi_2015_mothly_composit = ndvi_relevant_months_sections[0].aggregate_temporal_period(
    period="month",
    reducer="median"
)
#ndvi_2015_mothly_composit.download("~/ndvi_2015_mothly_composit.tif")
job = ndvi_2015_mothly_composit.create_job(
    out_format="GTiff",
    title="NDVI 2015 monthly",
    description="NDVI monthly composites for 2015 (Jun-Oct)"
)

job.start_and_wait()

results = job.get_results()
results.download_files("~/ndvi_2015/")

0:00:00 Job 'j-26042222045047629d5b5ff48f774cd1': send 'start'
0:00:47 Job 'j-26042222045047629d5b5ff48f774cd1': created (progress 0%)
0:00:52 Job 'j-26042222045047629d5b5ff48f774cd1': created (progress 0%)
0:00:59 Job 'j-26042222045047629d5b5ff48f774cd1': queued (progress 0%)
0:01:07 Job 'j-26042222045047629d5b5ff48f774cd1': queued (progress 0%)
0:01:17 Job 'j-26042222045047629d5b5ff48f774cd1': queued (progress 0%)
0:01:29 Job 'j-26042222045047629d5b5ff48f774cd1': queued (progress 0%)
0:01:44 Job 'j-26042222045047629d5b5ff48f774cd1': queued (progress 0%)
0:02:04 Job 'j-26042222045047629d5b5ff48f774cd1': queued (progress 0%)
0:02:28 Job 'j-26042222045047629d5b5ff48f774cd1': queued (progress 0%)
0:02:57 Job 'j-26042222045047629d5b5ff48f774cd1': queued (progress 0%)
0:03:35 Job 'j-26042222045047629d5b5ff48f774cd1': queued (progress 0%)
0:04:22 Job 'j-26042222045047629d5b5ff48f774cd1': queued (progress 0%)
0:05:20 Job 'j-26042222045047629d5b5ff48f774cd1': queued (progress 0%)
0:06:20 Job 

[PosixPath('~/ndvi_2015/openEO_2015-07-01Z.tif'),
 PosixPath('~/ndvi_2015/openEO_2015-08-01Z.tif'),
 PosixPath('~/ndvi_2015/openEO_2015-09-01Z.tif'),
 PosixPath('~/ndvi_2015/job-results.json')]

## ----------------------------------------------------------

In [1]:
# import necessary packages
import openeo
import openeo.processes as op
year_start = 2015
year_end = 2025

In [6]:
# connect with the backend
eoconn = openeo.connect(
        "openeo.dataspace.copernicus.eu"
        ).authenticate_oidc()
# Setup process parameters
bbox = {
        "west":  18.51635,
        "south": 48.78376,
        "east":  18.80255,
        "north": 49.04104
        }
date = f"{year_start}-05-05", f"{year_start}-12-31"

# Create a processing graph from the NDVI process using an active openEO connection
sentinel2_data_cube = eoconn.load_collection(
    "SENTINEL2_L2A",
    bands=["B04", "B08", "SCL"],
    spatial_extent=bbox,
    temporal_extent=["2020-06-01", "2020-10-01"],
)
sentinel2_data_cube.mask_clouds()

ndvi = sentinel2_data_cube.ndvi(red="B04",
    nir="B08")

Authenticated using refresh token.


AttributeError: 'DataCube' object has no attribute 'mask_clouds'